# X-Ray Analysis

Analysis of X-Ray images of batteries at varying conditions.

In [1]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import re
from dataclasses import dataclass
from typing import List, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed


XR_DATA_DIR = "../Data/XR"
PLOTS_DIR = "../Plots/XR"

BATTERIES = [f"C{i:02d}" for i in range(1, 15)]
POSITIONS = [
    "B1",
    "B2",
    "B3",
    "B4",
    "Position1",
    "Position2",
    "Position3",
    "Position4",
    "Position5",
    "Position6",
    "Position7",
    "Position8",
    "Position9",
    "Position10",
    "Position11",
    "Position12",
    "Position13",
    "Position14",
    "Tilt1",
    "Tilt2",
    "Tilt3",
    "Tilt4",
]

In [2]:
@dataclass
class ImageData:
    """
    Class to hold image data and its attributes.
    """

    image: np.ndarray
    state: str
    battery: str
    position: str

    def __post_init__(self):
        if not isinstance(self.image, np.ndarray):
            raise ValueError("Image must be a numpy array.")
        if not self.state or not self.battery or not self.position:
            raise ValueError("State, battery, and position must be defined.")


def load_image(filepath) -> ImageData:
    """
    Load an image from the specified file path.

    Args:
            filepath (str): The path to the image file.

    Returns:
            numpy.ndarray: The loaded image.
    """
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Image file not found: {filepath}")

    image = cv2.imread(filepath, cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError(f"Could not read image: {filepath}")

    if "Baseline" in filepath:
        state = "Baseline"
    elif "LT" in filepath:
        state = "LT"
    elif "HT-A" in filepath:
        state = "HT-A"
    elif "HT-B" in filepath:
        state = "HT-B"
    else:
        state = "Unknown"

    reBatteryMatch = re.search(r"C\d{2}", filepath)
    battery = reBatteryMatch.group(0) if reBatteryMatch else "Unknown"

    filename = os.path.basename(filepath)
    if "_" in filename:
        position = filename.split(".jpg")[0].split("_")[-1]
    else:
        position = filename.split(".jpg")[0]

    # Expand shortened position names used in baseline images
    if state == "Baseline":
        if re.search(r"P\d{1,2}", position):
            position = f"Position{position[1:]}"

    return ImageData(
        image=image,
        state=state,
        battery=battery,
        position=position,
    )


def find_images(directory: str) -> List[str]:
    """
    Find all image files in the specified directory.

    Args:
        directory (str): The directory to search for images.

    Returns:
        list: A list of file paths to the images.
    """
    if not os.path.exists(directory):
        raise FileNotFoundError(f"Directory not found: {directory}")

    image_files = []
    for root, _, files in os.walk(directory):
        for file in files:
            if file.lower().endswith(".jpg"):
                image_files.append(os.path.join(root, file))

    return image_files


def load_images(directory: str) -> List[ImageData]:
    """
    Load all images from the specified directory.

    Args:
        directory (str): The directory containing the images.

    Returns:
        list: A list of ImageData objects.
    """
    image_files = find_images(directory)
    images = []

    def safe_load_image(filepath):
        try:
            return load_image(filepath)
        except Exception as e:
            print(f"Error loading image {filepath}: {e}")
            return None

    with ThreadPoolExecutor() as executor:
        futures = [
            executor.submit(safe_load_image, filepath) for filepath in image_files
        ]
        for future in as_completed(futures):
            image_data = future.result()
            if image_data is not None:
                images.append(image_data)

    return images


images = load_images(XR_DATA_DIR)

In [3]:
# Filter images with an unknown battery or C00 (dummy battery)
images = [img for img in images if img.battery != "Unknown" and img.battery != "C00"]

## Collages

In [4]:
def collage_images(images: List[ImageData], grid: Tuple[int, int]):
    """
    Create a collage of images in a grid layout.

    Args:
        images (list): List of ImageData objects.
        grid (tuple): Tuple specifying the number of rows and columns in the grid.

    Returns:
        matplotlib.figure.Figure: The figure containing the collage.
    """
    rows, cols = grid
    if len(images) < rows * cols:
        raise ValueError("Not enough images to fill the grid.")
    else:
        # If there are fewer images than grid slots, fill the rest with blank white images
        img_shape = images[0].image.shape if images else (100, 100)
        blank_image = np.full(img_shape, 255, dtype=np.uint8)
        while len(images) < rows * cols:
            images.append(
                ImageData(image=blank_image, state="", battery="", position="")
            )

    # Determine the size of each image in inches (assuming 100 dpi for matplotlib)
    img_height, img_width = images[0].image.shape
    dpi = 330
    figsize = (cols * img_width / dpi, rows * img_height / dpi)

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=figsize,
        dpi=dpi,
    )
    axes = axes.flatten()  # Flatten the 2D array of axes for easy iteration

    for ax, image_data in zip(axes, images[: rows * cols]):
        ax.imshow(image_data.image, cmap="gray")
        ax.set_title(
            f"{image_data.state} - {image_data.battery} - {image_data.position}"
        )
        ax.axis("off")

    # Hide any unused subplots
    for ax in axes[len(images) :]:
        ax.axis("off")

    # plt.tight_layout()
    return fig

In [5]:
# Collage all images of the same battery and state
def group_images_by_battery_and_state(
    images: List[ImageData],
) -> List[Tuple[str, str, List[ImageData]]]:
    """
    Group images by battery and state.

    Args:
        images (list): List of ImageData objects.

    Returns:
        list: A list of tuples containing battery, state, and corresponding images.
    """
    grouped_images = {}
    for image_data in images:
        key = (image_data.battery, image_data.state)
        if key not in grouped_images:
            grouped_images[key] = []
        grouped_images[key].append(image_data)

    # Sort the images by position within each group
    for key, imgs in grouped_images.items():
        imgs.sort(key=lambda x: x.position)

    return [(battery, state, imgs) for (battery, state), imgs in grouped_images.items()]

In [6]:
def plot_images_by_battery_and_state(
    images: List[ImageData],
    grid: Tuple[int, int],
    output_dir: str = PLOTS_DIR,
) -> None:
    """
    Plot images grouped by battery and state.

    Args:
        images (list): List of ImageData objects.
        output_dir (str): Directory to save the plots.
        grid (tuple): Tuple specifying the number of rows and columns in the grid.
    """
    os.makedirs(output_dir, exist_ok=True)

    grouped_images = group_images_by_battery_and_state(images)

    def save_collage(battery, state, imgs):
        fig = collage_images(imgs, grid)
        filename = f"{battery}_{state}.png"

        for subdir in [battery, state]:
            subdir = os.path.join(output_dir, subdir)
            os.makedirs(subdir, exist_ok=True)
            fig.savefig(os.path.join(subdir, filename))

        plt.close(fig)

    with ThreadPoolExecutor() as executor:
        for battery, state, imgs in grouped_images:
            executor.submit(save_collage, battery, state, imgs)


plot_images_by_battery_and_state(
    images,
    grid=(4, 5),
    output_dir=os.path.join(PLOTS_DIR, "Collages"),
)

In [7]:
def group_images_by_battery_and_position(
    images: List[ImageData],
) -> List[Tuple[str, str, List[ImageData]]]:
    """
    Group images by battery and position.

    Args:
        images (list): List of ImageData objects.

    Returns:
        list: A list of tuples containing battery, position, and corresponding images.
    """
    grouped_images = {}
    for image_data in images:
        key = (image_data.battery, image_data.position)
        if key not in grouped_images:
            grouped_images[key] = []
        grouped_images[key].append(image_data)

    # Sort the images by state within each group
    for key, imgs in grouped_images.items():
        imgs.sort(key=lambda x: x.state)

    return [
        (battery, position, imgs)
        for (battery, position), imgs in grouped_images.items()
    ]


def plot_images_by_battery_and_position(
    images: List[ImageData],
    grid: Tuple[int, int],
    output_dir: str = PLOTS_DIR,
) -> None:
    """
    Plot images grouped by battery and position.

    Args:
        images (list): List of ImageData objects.
        output_dir (str): Directory to save the plots.
        grid (tuple): Tuple specifying the number of rows and columns in the grid.
    """
    os.makedirs(output_dir, exist_ok=True)

    grouped_images = group_images_by_battery_and_position(images)

    def save_collage(battery, position, imgs):
        fig = collage_images(imgs, grid)
        filename = f"{battery}_{position}.png"

        for subdir in [battery, position]:
            subdir = os.path.join(output_dir, subdir)
            os.makedirs(subdir, exist_ok=True)
            fig.savefig(os.path.join(subdir, filename))

        plt.close(fig)

    with ThreadPoolExecutor() as executor:
        for battery, position, imgs in grouped_images:
            executor.submit(save_collage, battery, position, imgs)


plot_images_by_battery_and_position(
    images,
    grid=(2, 2),
    output_dir=os.path.join(PLOTS_DIR, "Comparisons", "RawImages"),
)

## Lead Mask

In [8]:
def lead_mask(image: np.ndarray, threshold: int) -> np.ndarray:
    """
    Create a mask for lead in the image based on a threshold.

    Args:
        image (numpy.ndarray): The input image.
        threshold (int): The threshold value for lead detection.

    Returns:
        numpy.ndarray: A binary mask where lead is detected.
    """
    if not isinstance(image, np.ndarray):
        raise ValueError("Input must be a numpy array.")

    # Assert that the image is grayscale
    if len(image.shape) != 2:
        raise ValueError("Input image must be a grayscale image.")

    # Create a binary mask where pixel values are greater than the threshold
    mask = image > threshold

    # Convert the mask to uint8 type (0 or 255)
    mask = mask.astype(np.uint8) * 255

    # Optionally, apply morphological operations to clean up the mask
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    return mask

In [9]:
lead_masks = [
    ImageData(
        image=lead_mask(
            image_data.image,
            threshold=50,
        ),
        state=image_data.state,
        battery=image_data.battery,
        position=image_data.position,
    )
    for image_data in images
]

lead_masked_images = [
    ImageData(
        image=cv2.bitwise_and(
            image_data.image, image_data.image, mask=image_data.image
        ),
        state=image_data.state,
        battery=image_data.battery,
        position=image_data.position,
    )
    for image_data in lead_masks
]

In [ ]:
not_lead_masks = [
    ImageData(
        image=cv2.bitwise_not(image_data.image),
        state=image_data.state,
        battery=image_data.battery,
        position=image_data.position,
    )
    for image_data in lead_masked_images
]

not_lead_masked_images = [
    ImageData(
        image=cv2.bitwise_and(
            image_data.image, image_data.image, mask=image_data.image
        ),
        state=image_data.state,
        battery=image_data.battery,
        position=image_data.position,
    )
    for image_data in not_lead_masks
]

In [11]:
plot_images_by_battery_and_position(
    lead_masked_images,
    grid=(2, 2),
    output_dir=os.path.join(PLOTS_DIR, "Comparisons", "LeadMasks"),
)

In [ ]:
plot_images_by_battery_and_position(
    not_lead_masked_images,
    grid=(2, 2),
    output_dir=os.path.join(PLOTS_DIR, "Comparisons", "NotLeadMasks"),
)

KeyboardInterrupt: 

## Differences

In [ ]:
# Filter to only LT and HT-A images
lt_ht_images = [img for img in images if img.state in ["LT", "HT-A"]]


def plot_lt_ht_diff_images(images: List[ImageData], output_dir: str) -> None:
    """
    Plot the differences between LT and HT-A images.

    Args:
        images (list): List of ImageData objects.
        output_dir (str): Directory to save the plots.
    """
    # Group images by position
    grouped_images = group_images_by_battery_and_position(images)

    # Plot differences for each position
    for battery, position, imgs in grouped_images:
        lt_img = next((img for img in imgs if img.state == "LT"), None)
        ht_img = next((img for img in imgs if img.state == "HT-A"), None)

        assert (
            lt_img is not None and ht_img is not None
        ), f"No LT or HT-A images for {battery} at {position}"

        # Subtract images
        # Compute difference as int16 to capture negative values
        # diff = lt_img.image.astype(np.int16) - ht_img.image.astype(np.int16)

        # # Create an RGB image: positive (blue), negative (red), zero (black)
        # diff_image = np.zeros((*diff.shape, 3), dtype=np.uint8)
        # # Positive differences: blue channel
        # diff_image[..., 2] = np.clip(diff, 0, 255)
        # # Negative differences: red channel
        # diff_image[..., 0] = np.clip(-diff, 0, 255)

        diff_image = cv2.subtract(lt_img.image, ht_img.image)

        # Save the difference image
        plt.imshow(diff_image, cmap="gray")
        plt.title(f"LT - HT-A Difference - {battery} - {position}")
        plt.axis("off")
        plt.tight_layout()

        subdir = os.path.join(output_dir, lt_img.battery)
        os.makedirs(subdir, exist_ok=True)
        plt.savefig(os.path.join(subdir, f"lt_ht_diff_{battery}_{position}.png"))

        plt.close()


plot_lt_ht_diff_images(
    lt_ht_images,
    output_dir=os.path.join(PLOTS_DIR, "Differences LT-HT"),
)